In [1]:
import sys
#lets you access python's runtime environment
from pathlib import Path
#sys.path is a built in variable in the sys module and contains a list of directories that is seached through when you do an import
#so we are appending the src directory to that
sys.path.append(str(Path().resolve().parent / "src"))
import config

import pandas as pd
import numpy as np
import datetime

In [2]:
f_cleaned = pd.read_csv(config.PROJECT_ROOT/ "data" / "cleaned_f.csv")
m_cleaned = pd.read_csv(config.PROJECT_ROOT/ "data" / "cleaned_m.csv")

f_cleaned['Date'] = pd.to_datetime(f_cleaned['Date'], format = '%Y-%m-%d')
m_cleaned['Date'] = pd.to_datetime(m_cleaned['Date'], format = '%Y-%m-%d')
f_sorted = f_cleaned.sort_values(['Name', 'Year', 'Date'])
m_sorted = m_cleaned.sort_values(['Name', 'Year', 'Date'])


C:\Users\bnpar\AppData\Local\Temp\ipykernel_34384\3413725149.py:1: DtypeWarning: Columns (33,38) have mixed types. Specify dtype option on import or set low_memory=False.
  f_cleaned = pd.read_csv(config.PROJECT_ROOT/ "data" / "cleaned_f.csv")
C:\Users\bnpar\AppData\Local\Temp\ipykernel_34384\3413725149.py:2: DtypeWarning: Columns (33,38) have mixed types. Specify dtype option on import or set low_memory=False.
  m_cleaned = pd.read_csv(config.PROJECT_ROOT/ "data" / "cleaned_m.csv")


Time from first competition to end of year in consideration

In [3]:
f_sorted['TimeCompeting'] = pd.to_datetime(f_sorted['Year'].astype(str) + '-12-31') - f_sorted.groupby('Name')['Date'].transform('min')
f_sorted['TimeCompeting'] = f_sorted['TimeCompeting'].dt.days


In [4]:
bomb_squat = (
    ((f_sorted['Squat1Kg'] <= 0) | f_sorted['Squat1Kg'].isna()) &
    ((f_sorted['Squat2Kg'] <= 0) | f_sorted['Squat2Kg'].isna()) &
    ((f_sorted['Squat3Kg'] <= 0) | f_sorted['Squat3Kg'].isna()) & 
    ((f_sorted['Best3SquatKg'] <= 0) | f_sorted['Best3SquatKg'].isna())
)

bomb_bench = (
    ((f_sorted['Bench1Kg'] <= 0) | f_sorted['Bench1Kg'].isna()) &
    ((f_sorted['Bench2Kg'] <= 0) | f_sorted['Bench2Kg'].isna()) &
    ((f_sorted['Bench3Kg'] <= 0) | f_sorted['Bench3Kg'].isna()) & 
    ((f_sorted['Best3BenchKg'] <= 0) | f_sorted['Best3BenchKg'].isna())
)

bomb_deadlift = (
    ((f_sorted['Deadlift1Kg'] <= 0) | f_sorted['Deadlift1Kg'].isna()) &
    ((f_sorted['Deadlift2Kg'] <= 0) | f_sorted['Deadlift2Kg'].isna()) &
    ((f_sorted['Deadlift3Kg'] <= 0) | f_sorted['Deadlift3Kg'].isna()) & 
    ((f_sorted['Best3DeadliftKg'] <= 0) | f_sorted['Best3DeadliftKg'].isna())
)


f_sorted['BombOut'] = bomb_squat | bomb_bench | bomb_deadlift

Number of times a lifter has bombed out. <br>
Number of meets since last bombout, capped at 999. If lifter has never bombed out this is set to the maximum cap of 999.

In [5]:
####FIS THIS

f_sorted['NumberOfBombOuts'] = f_sorted.groupby('Name')['BombOut'].cumsum()
f_sorted['MeetsSinceLastBombOut'] = f_sorted.groupby(['Name', 'NumberOfBombOuts']).cumcount()

#past capped number of bombouts it gets set to cap
f_sorted.loc[f_sorted['MeetsSinceLastBombOut']> config.CAP_MEETS_SINCE_BOMBOUT, 'MeetsSinceLastBombOut'] = config.CAP_MEETS_SINCE_BOMBOUT

#if lifter has never bombed out number of meets since last bombout is set to cap
f_sorted.loc[(f_sorted['NumberOfBombOuts'] ==0 ), 'MeetsSinceLastBombOut'] = config.CAP_MEETS_SINCE_BOMBOUT



In [6]:
attempt_cols = ['Squat1Kg','Squat2Kg','Squat3Kg',
                   'Bench1Kg','Bench2Kg','Bench3Kg',
                   'Deadlift1Kg','Deadlift2Kg','Deadlift3Kg']

f_sorted['AttemptsMade'] = (f_sorted[attempt_cols]>0).sum(axis=1)

#handling the case where the lifter didnt bombout but it's recorded as 0 attempts because individual attempst werent filled in 
mask = (f_sorted['BombOut']==False)&(f_sorted['AttemptsMade']==0)

#excluding lifters who bombed out form calculation as we already know this lifter did not bomb out
mean_attempts = f_sorted.loc[(f_sorted['BombOut']==False) & (f_sorted['AttemptsMade']>0),'AttemptsMade'].mean().round() 

f_sorted.loc[mask, 'AttemptsMade'] = mean_attempts

In [7]:
f_sorted['LastMeetAttemptsMade'] = f_sorted.groupby(['Name', 'Year'])['AttemptsMade'].transform('last')
f_sorted['AverageAttemptsMade'] = f_sorted.groupby(['Name', 'Year'])['AttemptsMade'].transform('mean')
f_sorted['BestMeetOfYear'] = f_sorted.groupby(['Name', 'Year'])['TotalKg'].transform('max')


In [8]:
# f_sorted['MeetsSinceLastBombOut'] = f_sorted.groupby(['Name', 'Year'])['MeetsSinceLastBombOut'].transform('last')
# f_sorted['NumberOfBombOuts'] = f_sorted.groupby(['Name', 'Year'])['NumberOfBombOuts'].transform('last')
f_sorted[f_sorted['AttemptsMade'] ==0]
print(len(f_sorted))
print(len(f_sorted[f_sorted['AttemptsMade'] ==0])/len(f_sorted))

172388
0.007784764600784277


In [15]:
panel_data = f_sorted.groupby(['Name', 'Year']).tail(1).reset_index(drop = True)

cols =['Name','Year', 'WeightClassKg', 'Age', 'Division', 'TimeCompeting', 'Goodlift', 'BestMeetOfYear',  'AverageAttemptsMade', 'LastMeetAttemptsMade', 'MeetsSinceLastBombOut', 'NumberOfBombOuts']
panel_data = panel_data[cols]
panel_data = panel_data.sort_values(['Name', 'Year'])
panel_data['CompetesNextYear'] = (
    (panel_data['Name'] == panel_data['Name'].shift(-1)) & 
    (panel_data['Year'] +1 == panel_data['Year'].shift(-1))
)


41469

In [13]:
panel_data

,Name,Year,WeightClassKg,Age,Division,TimeCompeting,Goodlift,BestMeetOfYear,AverageAttemptsMade,LastMeetAttemptsMade,MeetsSinceLastBombOut,NumberOfBombOuts,CompetesNextYear
0,A Ajeesha,2017,72,16.5,Sub-Juniors,27,61.15,300.0,8.0,8,999,0,False
1,A Basma Ahmadi,2025,84,NaN,Juniors,181,58.11,300.0,7.0,7,999,0,False
2,A Madhuri,2022,76,14.5,Sub-Juniors,266,57.72,290.0,7.0,7,999,0,True
3,A Madhuri,2023,76,15.5,Sub-Juniors,631,56.94,287.5,7.0,7,999,0,True
4,A Madhuri,2024,76,16.5,Sub-Juniors,997,59.15,300.0,7.0,7,999,0,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...
110507,우연수,2024,57,NaN,Open,276,68.33,285.0,7.0,7,999,0,False
110508,이진화,2024,63,NaN,Open,276,58.96,262.5,7.0,7,999,0,False
110509,최자윤,2024,57,NaN,Open,276,62.03,262.5,6.0,6,999,0,False
110510,최향,2024,57,NaN,Open,276,55.06,230.0,3.0,3,999,0,False


In [ ]:
panel_data['Year'].dtype

In [ ]:
panel_data[panel_data['NumberOfBombOuts']>5]

In [ ]:
###EDA vibes
null_attempt = f_sorted[
    f_sorted['Squat1Kg'].isna() |
    f_sorted['Squat2Kg'].isna() |
    f_sorted['Squat3Kg'].isna() |
    f_sorted['Bench1Kg'].isna() |
    f_sorted['Bench2Kg'].isna() |
    f_sorted['Bench3Kg'].isna() |
    f_sorted['Deadlift1Kg'].isna() |
    f_sorted['Deadlift2Kg'].isna() |
    f_sorted['Deadlift3Kg'].isna()
]

all_attempts_null = f_sorted[
    f_sorted['Squat1Kg'].isna() &
    f_sorted['Squat2Kg'].isna() &
    f_sorted['Squat3Kg'].isna() &
    f_sorted['Bench1Kg'].isna() &
    f_sorted['Bench2Kg'].isna() &
    f_sorted['Bench3Kg'].isna() &
    f_sorted['Deadlift1Kg'].isna() &
    f_sorted['Deadlift2Kg'].isna() &
    f_sorted['Deadlift3Kg'].isna()
]

print(len(null_attempt)/len(f_sorted))
print(len(all_attempts_null)/ len(f_sorted))

failed_attempt = f_sorted[
    (f_sorted['Squat1Kg'] < 0) |
    (f_sorted['Squat2Kg'] < 0) |
    (f_sorted['Squat3Kg'] < 0) |
    (f_sorted['Bench1Kg'] < 0) |
    (f_sorted['Bench2Kg'] < 0) |
    (f_sorted['Bench3Kg'] < 0) |
    (f_sorted['Deadlift1Kg'] < 0) |
    (f_sorted['Deadlift2Kg'] < 0) |
    (f_sorted['Deadlift3Kg'] < 0)
]
